In [1]:
import pandas as pd

train = pd.read_csv("../house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("../house-prices-advanced-regression-techniques/test.csv")

train.head()


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Missing Values Overview


In [21]:
train.isnull().sum().sort_values(ascending=False)

PoolQC           1453
MiscFeature      1406
Alley            1369
Fence            1179
MasVnrType        872
                 ... 
MoSold              0
YrSold              0
SaleType            0
SaleCondition       0
SalePrice           0
Length: 81, dtype: int64

## Column Types: Categorical vs Numerical


In [22]:
from IPython.display import display

display(train.select_dtypes(include="object").columns)
display(train.select_dtypes(exclude="object").columns)



C:\Users\cicar\AppData\Local\Temp\ipykernel_3988\869671163.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(train.select_dtypes(include="object").columns)


Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='str')

Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
       'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF',
       'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea',
       'MiscVal', 'MoSold', 'YrSold', 'SalePrice'],
      dtype='str')

# Missing value summary

In [23]:
missing_summary = train.isna().sum().sort_values(ascending=False)
missing_percentage = (train.isna().mean() * 100).sort_values(ascending=False)

pd.DataFrame({
    'missing_count': missing_summary,
    'missing_percentage': missing_percentage.round(2)
})

,missing_count,missing_percentage
PoolQC,1453,99.52
MiscFeature,1406,96.30
Alley,1369,93.77
Fence,1179,80.75
MasVnrType,872,59.73
...,...,...
MoSold,0,0.00
YrSold,0,0.00
SaleType,0,0.00
SaleCondition,0,0.00


### Drop High-Missing Columns (>80%)

The columns PoolQC (99.5%), MiscFeature (96.3%), Alley (93.8%), and Fence (80.7%)
were already removed earlier in the workflow or were not present in the loaded dataset.

These features contain extremely sparse information and provide no predictive value,
so no further action is required at this step.



In [2]:
cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence']


### Impute Remaining Missing Values

- MasVnrType: Missing values indicate no masonry veneer → imputed with "None".
- LotFrontage: Numeric and skewed → imputed with median.
- GarageType, GarageFinish, GarageQual, GarageCond: Missing values indicate no garage → imputed with "None".
- BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1, BsmtFinType2: Missing values indicate no basement → imputed with "None".
- Electrical: Very few missing values → imputed with mode.


In [3]:
# MasVnrType: missing means no masonry veneer
train['MasVnrType'] = train['MasVnrType'].fillna('None')

# LotFrontage: numeric + skewed → median
train['LotFrontage'] = train['LotFrontage'].fillna(train['LotFrontage'].median())

# Garage-related: missing means no garage
garage_cols = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']
for col in garage_cols:
    train[col] = train[col].fillna('None')

# Basement-related: missing means no basement
bsmt_cols = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
for col in bsmt_cols:
    train[col] = train[col].fillna('None')

# Electrical: few missing → mode
train['Electrical'] = train['Electrical'].fillna(train['Electrical'].mode()[0])


## Outlier Detection

To identify extreme numeric values that may distort model performance, we inspect
the distribution of key continuous features using boxplots. Outliers will be
evaluated individually and removed only when they represent data entry errors
or unrealistic values.


In [4]:
import matplotlib.pyplot as plt
import seaborn as sns

numeric_cols = [
    'LotFrontage', 'LotArea', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2',
    'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
    'GrLivArea', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch',
    '3SsnPorch', 'ScreenPorch', 'PoolArea'
]

plt.figure(figsize=(16, 40))
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(len(numeric_cols), 1, i)
    sns.boxplot(x=train[col], color='skyblue')
    plt.title(col)

plt.tight_layout()
plt.show()


ModuleNotFoundError: No module named 'matplotlib'

## Outlier Removal

Based on the boxplots, several numeric features contain extreme values that are
not representative of typical residential properties. These outliers can distort
model training and reduce predictive performance. We remove only the most
well‑known extreme cases documented in the Ames dataset, focusing on unrealistic
values rather than legitimate large homes.

The following outlier rules are applied:

- Remove properties with extremely large above‑ground living area (GrLivArea > 4000)
- Remove properties with excessively large lot size (LotArea > 100000)
- Remove properties with unusually large total basement area (TotalBsmtSF > 3000)
- Remove properties with oversized garage area (GarageArea > 1200)

These thresholds follow common preprocessing practice for the Ames dataset and
help stabilize model training without removing valid high‑value homes.


In [5]:
# Outlier removal rules based on known Ames dataset extremes

train_clean = train.copy()

train_clean = train_clean[
    (train_clean['GrLivArea'] <= 4000) &
    (train_clean['LotArea'] <= 100000) &
    (train_clean['TotalBsmtSF'] <= 3000) &
    (train_clean['GarageArea'] <= 1200)
]

print("Original rows:", len(train))
print("Cleaned rows:", len(train_clean))
print("Rows removed:", len(train) - len(train_clean))


Original rows: 1460
Cleaned rows: 1445
Rows removed: 15


## Feature Engineering

To improve model performance, several numeric features are combined or transformed
to better represent property characteristics. Feature engineering focuses on
creating meaningful aggregated values and ensuring all inputs are numeric and
consistent for model training.

The following engineered features are added:

- TotalSF: combined square footage of basement, first floor, and second floor
- TotalPorchSF: sum of all porch-related square footage
- AgeAtSale: property age at the time of sale
- TotalBath: combined full and half bathrooms

These engineered features help the model capture overall size, age, and amenity
levels more effectively than using raw columns alone.


In [6]:
# Feature Engineering

train_fe = train_clean.copy()

# Total square footage
train_fe['TotalSF'] = (
    train_fe['TotalBsmtSF'] +
    train_fe['1stFlrSF'] +
    train_fe['2ndFlrSF']
)

# Total porch square footage
train_fe['TotalPorchSF'] = (
    train_fe['OpenPorchSF'] +
    train_fe['EnclosedPorch'] +
    train_fe['3SsnPorch'] +
    train_fe['ScreenPorch']
)

# Age at sale
train_fe['AgeAtSale'] = train_fe['YrSold'] - train_fe['YearBuilt']

# Total bathrooms
train_fe['TotalBath'] = (
    train_fe['FullBath'] +
    (0.5 * train_fe['HalfBath']) +
    train_fe['BsmtFullBath'] +
    (0.5 * train_fe['BsmtHalfBath'])
)

train_fe.head()


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,TotalSF,TotalPorchSF,AgeAtSale,TotalBath
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,2,2008,WD,Normal,208500,2566,61,5,3.5
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,5,2007,WD,Normal,181500,2524,0,31,2.5
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,9,2008,WD,Normal,223500,2706,42,7,3.5
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,2,2006,WD,Abnorml,140000,2473,307,91,2.0
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,12,2008,WD,Normal,250000,3343,84,8,3.5


## Model Training

A RandomForestRegressor is used as the primary model for predicting house prices.
This algorithm performs well on tabular datasets, handles non-linear relationships,
and is robust to moderate outliers and feature scaling differences.

The dataset is split into training and validation sets to evaluate performance.
After training, the model is saved as `model.pkl` for use in the Streamlit
application.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import joblib

# Select features (all numeric columns except target)
X = train_fe.drop(columns=['SalePrice'])
y = train_fe['SalePrice']

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    random_state=42
)

model.fit(X_train, y_train)

# Validation
preds = model.predict(X_val)
mae = mean_absolute_error(y_val, preds)

print("Validation MAE:", mae)

# Save model
joblib.dump(model, "model.pkl")

print("model.pkl saved successfully.")
